# Rademacher Complexity

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 13.05, 09.04 |
| Optional | 01.01 (vectors — for linear class computation) |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/06_rademacher_complexity.ipynb)

---

## 🎯 Learning Objective

Understand **Rademacher complexity** as a data-dependent measure of function-class richness, derive the Rademacher generalization bound via symmetrization and McDiarmid's inequality (from `09.04`), compute Rademacher complexity for linear classifiers and neural networks through the contraction lemma, and compare Rademacher-based bounds to VC-dimension bounds (from `13.05`).

---

## 📐 Theory

VC dimension (`13.05`) gives a *worst-case*, data-independent measure of hypothesis-class capacity. **Rademacher complexity** provides a strictly finer, *data-dependent* alternative: it measures how well a function class can correlate with random noise on the *specific* sample at hand, yielding tighter generalization bounds whenever the data is "well-behaved" relative to the class.

### Empirical Rademacher Complexity

Let $\mathcal{F}$ be a class of real-valued functions $f: \mathcal{Z} \to \mathbb{R}$, and let $S = (z_1, \ldots, z_n)$ be a fixed sample drawn from distribution $\mathcal{D}$. The **empirical Rademacher complexity** of $\mathcal{F}$ with respect to $S$ is:

$$\hat{\mathcal{R}}_S(\mathcal{F}) \;=\; \mathbb{E}_{\boldsymbol{\sigma}}\!\left[\sup_{f \in \mathcal{F}} \frac{1}{n} \sum_{i=1}^{n} \sigma_i f(z_i)\right]$$

where $\sigma_1, \ldots, \sigma_n$ are independent **Rademacher random variables**: each $\sigma_i \in \{-1, +1\}$ with equal probability $1/2$.

**Symbol definitions:**
- $\mathcal{F}$: the function class (hypothesis space) under analysis.
- $S = (z_1, \ldots, z_n)$: the observed sample of size $n$.
- $\sigma_i$: i.i.d. Rademacher random variables (uniform on $\{-1, +1\}$).
- $\sup_{f \in \mathcal{F}}$: supremum over all functions in the class.

**Intuition:** The Rademacher variables $\sigma_i$ represent *pure random noise* — a completely random labeling. The inner supremum asks: *"What is the best correlation any $f \in \mathcal{F}$ can achieve with this random noise?"* A rich function class can fit even random noise well, so $\hat{\mathcal{R}}_S(\mathcal{F})$ is large. A restricted class cannot, so $\hat{\mathcal{R}}_S(\mathcal{F})$ is small. This directly measures the class's capacity to overfit.

### Rademacher Complexity (Population Version)

The **Rademacher complexity** averages the empirical version over all possible samples of size $n$:

$$\mathcal{R}_n(\mathcal{F}) \;=\; \mathbb{E}_{S \sim \mathcal{D}^n}\!\left[\hat{\mathcal{R}}_S(\mathcal{F})\right]$$

This removes dependence on the particular sample $S$, giving a distributional quantity analogous to VC dimension but still distribution-dependent (and therefore potentially tighter).

### Rademacher Generalization Bound

The central result connecting Rademacher complexity to generalization. For any class $\mathcal{F}$ of functions mapping into $[0,1]$, with probability at least $1-\delta$ over a sample $S$ of size $n$:

$$\sup_{f \in \mathcal{F}} \left| R(f) - \hat{R}(f) \right| \;\leq\; 2\,\mathcal{R}_n(\mathcal{F}) + \sqrt{\frac{\ln(2/\delta)}{2n}}$$

where $R(f) = \mathbb{E}_{z \sim \mathcal{D}}[f(z)]$ is the true risk and $\hat{R}(f) = \frac{1}{n}\sum_{i=1}^{n} f(z_i)$ is the empirical risk.

**Proof sketch (symmetrization + McDiarmid):**

1. **Symmetrization.** Introduce a *ghost sample* $S' = (z_1', \ldots, z_n')$, also i.i.d. from $\mathcal{D}$. Since $\mathbb{E}_{S'}[\hat{R}_{S'}(f)] = R(f)$ for each fixed $f$, we can write:

$$\mathbb{E}_S\!\left[\sup_{f \in \mathcal{F}} \left(R(f) - \hat{R}(f)\right)\right] \;\leq\; \mathbb{E}_{S, S'}\!\left[\sup_{f \in \mathcal{F}} \frac{1}{n}\sum_{i=1}^{n}\left(f(z_i') - f(z_i)\right)\right]$$

2. **Rademacher injection.** Since each pair $(z_i, z_i')$ is exchangeable, introducing $\sigma_i \in \{-1,+1\}$ to randomly swap them does not change the expectation:

$$= \mathbb{E}_{S, S', \boldsymbol{\sigma}}\!\left[\sup_{f \in \mathcal{F}} \frac{1}{n}\sum_{i=1}^{n} \sigma_i\left(f(z_i') - f(z_i)\right)\right] \;\leq\; 2\,\mathcal{R}_n(\mathcal{F})$$

   The last step uses the triangle inequality and the fact that $\sup_f \frac{1}{n}\sum \sigma_i f(z_i)$ appears for both $S$ and $S'$.

3. **Concentration via McDiarmid.** The function $\phi(S) = \sup_{f \in \mathcal{F}} (R(f) - \hat{R}(f))$ satisfies bounded differences: changing any single $z_i$ changes $\phi$ by at most $1/n$ (since $f$ maps to $[0,1]$). McDiarmid's inequality (recall `09.04`) then gives:

$$P\!\left(\phi(S) - \mathbb{E}[\phi(S)] \geq t\right) \leq \exp\!\left(-2nt^2\right)$$

   Setting $\delta = 2\exp(-2nt^2)$ and solving for $t$ yields the $\sqrt{\ln(2/\delta)/(2n)}$ tail term, completing the bound.

**Data-dependent variant.** Replacing $\mathcal{R}_n(\mathcal{F})$ with the empirical $\hat{\mathcal{R}}_S(\mathcal{F})$ in the bound is also valid (with a slightly adjusted confidence term), since McDiarmid also controls the deviation of $\hat{\mathcal{R}}_S$ from $\mathcal{R}_n$.

### Rademacher Complexity of Specific Classes

#### Linear classifiers with bounded norm

Consider $\mathcal{F} = \{x \mapsto \langle w, x \rangle : \|w\| \leq B\}$ for data $x_1, \ldots, x_n \in \mathbb{R}^d$. Then:

$$\hat{\mathcal{R}}_S(\mathcal{F}) \;\leq\; \frac{B \cdot \max_i \|x_i\|}{\sqrt{n}}$$

**Derivation:** By definition,

$$\hat{\mathcal{R}}_S(\mathcal{F}) = \mathbb{E}_{\boldsymbol{\sigma}}\!\left[\sup_{\|w\| \leq B} \frac{1}{n} \sum_{i=1}^{n} \sigma_i \langle w, x_i \rangle\right] = \mathbb{E}_{\boldsymbol{\sigma}}\!\left[\sup_{\|w\| \leq B} \frac{1}{n} \left\langle w, \sum_{i=1}^{n} \sigma_i x_i \right\rangle\right]$$

By Cauchy–Schwarz (recall `01.01`), $\sup_{\|w\| \leq B} \langle w, v \rangle = B\|v\|$, so:

$$= \frac{B}{n}\, \mathbb{E}_{\boldsymbol{\sigma}}\!\left[\left\|\sum_{i=1}^{n} \sigma_i x_i\right\|\right] \;\leq\; \frac{B}{n}\sqrt{\mathbb{E}\!\left[\left\|\sum_{i=1}^{n} \sigma_i x_i\right\|^2\right]}$$

by Jensen's inequality. Expanding the squared norm and using $\mathbb{E}[\sigma_i \sigma_j] = \delta_{ij}$:

$$= \frac{B}{n}\sqrt{\sum_{i=1}^{n} \|x_i\|^2} \;\leq\; \frac{B}{n}\sqrt{n \cdot \max_i \|x_i\|^2} \;=\; \frac{B \cdot \max_i \|x_i\|}{\sqrt{n}}$$

This bound scales as $O(1/\sqrt{n})$ regardless of the dimension $d$ — a crucial property that makes norm-based generalization bounds meaningful even in very high-dimensional spaces.

#### Neural networks: spectral-norm bound via contraction

For a depth-$L$ feedforward network with weight matrices $W_1, \ldots, W_L$ and $1$-Lipschitz activation $\phi$ (e.g., ReLU), the Rademacher complexity satisfies:

$$\mathcal{R}_n(\mathcal{F}_{\text{net}}) \;\leq\; \frac{\left(\prod_{\ell=1}^{L} \|W_\ell\|_{\text{op}}\right) \cdot \max_i \|x_i\|}{\sqrt{n}}$$

where $\|W_\ell\|_{\text{op}}$ is the spectral norm (largest singular value) of the $\ell$-th weight matrix. This is derived by applying the **contraction lemma** layer by layer, peeling off each activation and weight matrix.

### Contraction Lemma (Talagrand / Ledoux–Talagrand)

If $\varphi: \mathbb{R} \to \mathbb{R}$ is $L$-Lipschitz with $\varphi(0) = 0$, and $\mathcal{F}$ is a function class, then:

$$\hat{\mathcal{R}}_S(\varphi \circ \mathcal{F}) \;\leq\; L \cdot \hat{\mathcal{R}}_S(\mathcal{F})$$

where $\varphi \circ \mathcal{F} = \{z \mapsto \varphi(f(z)) : f \in \mathcal{F}\}$.

**Proof sketch:** For each fixed $\boldsymbol{\sigma}$, write $g^* = \arg\sup_{f \in \mathcal{F}} \frac{1}{n}\sum_i \sigma_i \varphi(f(z_i))$. Since $\varphi$ is $L$-Lipschitz and $\varphi(0)=0$, we have $|\varphi(u)| \leq L|u|$, so:

$$\frac{1}{n}\sum_{i=1}^{n} \sigma_i \varphi(g^*(z_i)) \;\leq\; \frac{L}{n}\sum_{i=1}^{n} |\sigma_i| \cdot |g^*(z_i)| \;\leq\; \frac{L}{n}\sup_{f \in \mathcal{F}} \sum_{i=1}^{n} \sigma_i f(z_i)$$

The rigorous argument uses a more careful comparison inequality (Ledoux–Talagrand), but the key message is: **composing with a Lipschitz function can only shrink (or at worst preserve) Rademacher complexity.** This is the fundamental tool for bounding the complexity of deep networks layer by layer.

**Application to neural networks:** ReLU is $1$-Lipschitz with $\text{ReLU}(0)=0$. Multiplication by $W_\ell$ scales Rademacher complexity by at most $\|W_\ell\|_{\text{op}}$. Composing $L$ layers gives the product $\prod_\ell \|W_\ell\|_{\text{op}}$.

### Rademacher Complexity vs. VC Dimension

| Property | VC dimension (`13.05`) | Rademacher complexity |
|:---|:---|:---|
| Data dependence | None (worst-case over all samples) | Yes (depends on actual data distribution) |
| Tightness | Can be very loose | Never looser than VC (up to constants), often tighter |
| Computability | Often combinatorial; NP-hard in general | Monte Carlo estimation from data |
| Function type | Binary classifiers ($\{0,1\}$) | Real-valued functions |
| Deep networks | VC bounds often vacuous | Spectral-norm bounds non-vacuous for moderate nets |

The relationship is formally captured by: for binary classifiers with VC dimension $d_{\text{VC}}$,

$$\mathcal{R}_n(\mathcal{F}) \;\leq\; \sqrt{\frac{2 d_{\text{VC}} \ln(en/d_{\text{VC}})}{n}}$$

so Rademacher complexity is always at most of order $\sqrt{d_{\text{VC}} \log n / n}$, but can be *much* smaller when the data has favorable structure. However, for large modern networks, even Rademacher-based bounds are typically *vacuous* (i.e., the bound exceeds $1$ for $0$-$1$ loss), motivating ongoing research into tighter complexity measures.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

We estimate the empirical Rademacher complexity $\hat{\mathcal{R}}_S(\mathcal{F})$ via Monte Carlo for function classes of **increasing richness** — from a single constant function up to an unrestricted finite class — and watch the complexity grow. We also overlay the VC-based generalization bound against the Rademacher-based bound and the true (simulated) generalization gap, showing how the Rademacher bound tracks the actual gap more tightly than VC.

In [ ]:
rng = np.random.default_rng(42)
n = 100  # sample size
d = 10   # input dimension
n_sigma = 2000  # Monte Carlo draws for Rademacher estimation

# Generate a fixed sample S of n points in R^d
X = rng.standard_normal((n, d))
X = X / np.linalg.norm(X, axis=1, keepdims=True)  # normalize to unit sphere

def mc_rademacher_linear(X, B, n_sigma=2000, rng=None):
    """Monte Carlo estimate of empirical Rademacher complexity for
    linear class {x -> <w,x> : ||w|| <= B}.
    Exact inner sup: B * ||sum sigma_i x_i|| (via Cauchy-Schwarz)."""
    rng = rng or np.random.default_rng()
    n = X.shape[0]
    sups = np.zeros(n_sigma)
    for t in range(n_sigma):
        sigma = rng.choice([-1, 1], size=n)
        v = (sigma[:, None] * X).sum(axis=0)  # sum_i sigma_i x_i
        sups[t] = B * np.linalg.norm(v) / n
    return sups.mean()

# --- Panel 1: Rademacher complexity vs. class richness (norm bound B) ---
B_values = np.array([0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
rad_estimates = [mc_rademacher_linear(X, B, n_sigma, rng) for B in B_values]
theoretical_upper = [B * 1.0 / np.sqrt(n) for B in B_values]  # max||x_i||=1

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
ax.loglog(B_values, rad_estimates, 'o-', color="#4C72B0", lw=2, markersize=6,
          label=r"Monte Carlo $\hat{\mathcal{R}}_S$ (linear class)")
ax.loglog(B_values, theoretical_upper, 's--', color="#C44E52", lw=2, markersize=5,
          label=r"Upper bound $B \cdot \max\|x_i\| / \sqrt{n}$")
ax.set_xlabel("Norm bound B (class richness)", fontsize=12)
ax.set_ylabel(r"$\hat{\mathcal{R}}_S(\mathcal{F})$", fontsize=12)
ax.set_title("Rademacher complexity grows\nwith class richness", fontsize=13)
ax.legend(fontsize=10)

# --- Panel 2: VC bound vs Rademacher bound vs true gap ---
n_values = np.arange(50, 1001, 50)
delta = 0.05
d_vc = d  # VC dim of linear classifiers in R^d

vc_bounds = []
rad_bounds = []
true_gaps = []

for nn in n_values:
    # VC generalization bound: sqrt(d_vc * ln(2en/d_vc) / n) + sqrt(ln(2/delta)/(2n))
    vc_b = np.sqrt(d_vc * np.log(2 * np.e * nn / d_vc) / nn) + np.sqrt(np.log(2/delta)/(2*nn))
    vc_bounds.append(vc_b)

    # Rademacher bound: 2*R_n + tail
    X_tmp = rng.standard_normal((nn, d))
    X_tmp = X_tmp / np.linalg.norm(X_tmp, axis=1, keepdims=True)
    rad_est = mc_rademacher_linear(X_tmp, B=1.0, n_sigma=500, rng=rng)
    rad_b = 2 * rad_est + np.sqrt(np.log(2/delta)/(2*nn))
    rad_bounds.append(rad_b)

    # True gap simulation: generate binary classification, fit best linear, measure gap
    w_true = rng.standard_normal(d)
    w_true = w_true / np.linalg.norm(w_true)
    y_train = (X_tmp @ w_true > 0).astype(float)
    # Best linear fit (least squares) with norm constraint
    w_hat = np.linalg.lstsq(X_tmp, y_train, rcond=None)[0]
    w_hat = w_hat / max(np.linalg.norm(w_hat), 1.0)  # project to ||w||<=1
    train_err = np.mean((X_tmp @ w_hat > 0).astype(float) != y_train)
    # Test on fresh data
    X_test = rng.standard_normal((5000, d))
    X_test = X_test / np.linalg.norm(X_test, axis=1, keepdims=True)
    y_test = (X_test @ w_true > 0).astype(float)
    test_err = np.mean((X_test @ w_hat > 0).astype(float) != y_test)
    true_gaps.append(abs(test_err - train_err))

ax = axes[1]
ax.plot(n_values, vc_bounds, 's-', color="#8172B2", lw=2, markersize=4, label="VC bound")
ax.plot(n_values, rad_bounds, 'o-', color="#C44E52", lw=2, markersize=4, label="Rademacher bound")
ax.plot(n_values, true_gaps, 'D-', color="#55A868", lw=2, markersize=4, label="True |test − train| gap")
ax.set_xlabel("Sample size n", fontsize=12)
ax.set_ylabel("Generalization gap", fontsize=12)
ax.set_title("Rademacher bound tracks the true gap\nmore tightly than VC", fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Left: Rademacher complexity scales linearly with B (norm bound = class richness), ")
print(f"      and Monte Carlo estimates sit below the theoretical upper bound.")
print(f"Right: As n grows, all three decrease, but the Rademacher bound is consistently")
print(f"       tighter than the VC bound, while the true gap is tightest of all.")

## 🐍 Implementation from Scratch

We implement a general-purpose **Monte Carlo Rademacher complexity estimator**: sample Rademacher vectors $\boldsymbol{\sigma}$, solve the inner supremum (analytically for linear classes, by enumeration for finite classes), and average. We then use the Rademacher generalization bound to produce a concrete complexity certificate, and compare it against the empirical generalization gap.

In [ ]:
def empirical_rademacher_linear(X, B, n_draws=5000, seed=42):
    """Monte Carlo estimate of empirical Rademacher complexity for
    the linear class F = {x -> <w,x> : ||w|| <= B}.

    Inner sup has closed form: B * ||sum_i sigma_i x_i|| / n
    (Cauchy-Schwarz, as derived in Theory section).
    """
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    sups = np.zeros(n_draws)
    for t in range(n_draws):
        sigma = rng.choice([-1, 1], size=n)
        v = sigma @ X  # sum_i sigma_i x_i, shape (d,)
        sups[t] = B * np.linalg.norm(v) / n
    return sups.mean(), sups.std() / np.sqrt(n_draws)


def empirical_rademacher_finite(F_vals, n_draws=5000, seed=42):
    """Monte Carlo estimate for a finite function class.

    F_vals: array of shape (|F|, n) — each row is (f(z_1), ..., f(z_n)) for one f in F.
    Inner sup by enumeration over the finite class.
    """
    rng = np.random.default_rng(seed)
    n = F_vals.shape[1]
    sups = np.zeros(n_draws)
    for t in range(n_draws):
        sigma = rng.choice([-1, 1], size=n)
        correlations = F_vals @ sigma / n  # shape (|F|,)
        sups[t] = correlations.max()
    return sups.mean(), sups.std() / np.sqrt(n_draws)


def rademacher_gen_bound(rad_complexity, n, delta=0.05):
    """Rademacher generalization bound:
    sup|R(f) - R_hat(f)| <= 2 * R_n(F) + sqrt(ln(2/delta) / (2n))
    """
    return 2 * rad_complexity + np.sqrt(np.log(2/delta) / (2*n))


# --- Demo 1: Linear class on synthetic data ---
rng = np.random.default_rng(0)
n, d = 200, 20
X = rng.standard_normal((n, d))
X = X / np.linalg.norm(X, axis=1, keepdims=True)  # unit sphere

for B in [0.5, 1.0, 5.0]:
    rad_est, rad_se = empirical_rademacher_linear(X, B)
    theory_ub = B * 1.0 / np.sqrt(n)  # max||x_i|| = 1
    gen_bound = rademacher_gen_bound(rad_est, n)
    print(f"B={B:5.1f}  |  R_hat(F) = {rad_est:.4f} ± {rad_se:.4f}  |  "
          f"Theory UB = {theory_ub:.4f}  |  Gen bound = {gen_bound:.4f}")

print()

# --- Demo 2: Finite class — increasing |F| ---
print("Finite class: Rademacher complexity grows with |F|")
for k in [2, 10, 50, 200]:
    # k random functions mapping n points to {0, 1}
    F_vals = rng.choice([0.0, 1.0], size=(k, n))
    rad_est, rad_se = empirical_rademacher_finite(F_vals)
    print(f"  |F| = {k:4d}  |  R_hat(F) = {rad_est:.4f} ± {rad_se:.4f}")

print()

# --- Demo 3: Full pipeline — compare bound to true gap ---
w_true = rng.standard_normal(d)
w_true = w_true / np.linalg.norm(w_true)
y = (X @ w_true > 0).astype(float)

# Fit linear classifier (OLS + norm projection)
w_hat = np.linalg.lstsq(X, y, rcond=None)[0]
w_hat = w_hat / max(np.linalg.norm(w_hat), 1.0)  # ||w|| <= 1

train_err = np.mean((X @ w_hat > 0).astype(float) != y)
X_test = rng.standard_normal((10000, d))
X_test = X_test / np.linalg.norm(X_test, axis=1, keepdims=True)
y_test = (X_test @ w_true > 0).astype(float)
test_err = np.mean((X_test @ w_hat > 0).astype(float) != y_test)

rad_est, _ = empirical_rademacher_linear(X, B=1.0)
gen_bound = rademacher_gen_bound(rad_est, n)
actual_gap = abs(test_err - train_err)

print(f"Train error:       {train_err:.4f}")
print(f"Test error:        {test_err:.4f}")
print(f"Actual gap:        {actual_gap:.4f}")
print(f"Rademacher bound:  {gen_bound:.4f}")
print(f"Bound valid: {actual_gap <= gen_bound}  (actual gap < Rademacher bound)")

## 🤖 Why This Matters for AI

Rademacher complexity provides the theoretical foundation for **spectral-norm regularization** — a widely used regularization technique in modern deep learning, and the basis of the spectrally-normalized generalization bounds of Bartlett, Foster & Telgarsky (2017) and Neyshabur et al. (2018) (both cited in Further Reading):

1. **Spectral normalization in GANs.** The discriminator in Spectral Normalization GAN (SNGAN) explicitly constrains each weight matrix's spectral norm $\|W_\ell\|_{\text{op}} \leq 1$, directly controlling the product $\prod_\ell \|W_\ell\|_{\text{op}}$ that appears in the Rademacher bound for neural networks. This isn't just an ad-hoc trick — it's a principled consequence of the theory above: smaller spectral norms → smaller Rademacher complexity → tighter generalization guarantee.

2. **Norm-based generalization bounds.** For moderately-sized networks, norm-based bounds like these can be far more informative than parameter-counting (VC-dimension) bounds, which are vacuous for over-parameterized networks (recall `13.05`). Whether they are *non-vacuous* (predict a test-error bound below $1$) depends on the network and data; the first demonstrably non-vacuous bounds for such networks came from a PAC-Bayes analysis (Dziugaite & Roy, 2017), and norm-based and PAC-Bayes bounds are closely related (Neyshabur et al., 2018) rather than one strictly dominating the other. The honest summary: norm-based complexity captures something parameter counts miss, but a bound that is both tight and non-vacuous at realistic scale remains open (see point 4).

3. **Weight decay as implicit Rademacher control.** Standard $L_2$ weight decay (regularizing $\sum_\ell \|W_\ell\|_F^2$) indirectly controls spectral norms and hence Rademacher complexity. The theoretical connection explains *why* weight decay helps generalization, beyond the informal "keeps weights small" intuition.

4. **Limits for LLMs.** For large language models with billions of parameters, even Rademacher bounds become vacuous — the product of spectral norms across hundreds of layers is astronomically large. This is an active research frontier: finding complexity measures that are both theoretically principled and practically non-vacuous at LLM scale.

Below, we demonstrate the spectral-norm / Rademacher connection concretely: we train small neural networks with and without spectral-norm constraints and verify the theoretical prediction that constraining spectral norms reduces the Rademacher complexity estimate and the actual generalization gap.

In [ ]:
def relu(x):
    return np.maximum(0, x)

def forward_net(X, weights):
    """Forward pass through a multi-layer ReLU network.
    weights: list of (W, b) tuples."""
    h = X
    for i, (W, b) in enumerate(weights):
        h = h @ W + b
        if i < len(weights) - 1:  # ReLU on all but last layer
            h = relu(h)
    return h.squeeze()

def spectral_norm_product(weights):
    """Product of spectral norms across all layers."""
    prod = 1.0
    for W, b in weights:
        prod *= np.linalg.svd(W, compute_uv=False)[0]  # largest singular value
    return prod

def make_random_net(dims, scale=1.0, seed=0):
    """Create a random ReLU network with given layer dimensions."""
    rng = np.random.default_rng(seed)
    weights = []
    for i in range(len(dims)-1):
        fan_in = dims[i]
        W = rng.standard_normal((dims[i], dims[i+1])) * scale / np.sqrt(fan_in)
        b = np.zeros(dims[i+1])
        weights.append((W, b))
    return weights

def project_spectral_norm(weights, max_norm=1.0):
    """Project each weight matrix to have spectral norm <= max_norm."""
    projected = []
    for W, b in weights:
        U, S, Vt = np.linalg.svd(W, full_matrices=False)
        S_clipped = np.minimum(S, max_norm)
        W_proj = U @ np.diag(S_clipped) @ Vt
        projected.append((W_proj, b))
    return projected

def mc_rademacher_network(X, weights, n_draws=2000, seed=42):
    """Monte Carlo Rademacher estimate for a network class.
    Uses the network output as a single function and estimates
    E_sigma[|1/n sum sigma_i f(x_i)|] as a lower bound on R_hat."""
    rng = np.random.default_rng(seed)
    f_vals = forward_net(X, weights)
    n = len(f_vals)
    sups = np.zeros(n_draws)
    for t in range(n_draws):
        sigma = rng.choice([-1, 1], size=n)
        sups[t] = abs(np.dot(sigma, f_vals)) / n
    return sups.mean()

# --- Setup ---
rng = np.random.default_rng(123)
n_train, n_test, d = 200, 2000, 10
X_train = rng.standard_normal((n_train, d))
X_test = rng.standard_normal((n_test, d))

# True labeling function (nonlinear)
w_true = rng.standard_normal(d)
y_train = (np.sin(X_train @ w_true) > 0).astype(float)
y_test = (np.sin(X_test @ w_true) > 0).astype(float)

# Compare: unconstrained vs. spectrally-normalized networks
print(f"{'Config':<25} {'Spec-norm prod':>15} {'Train err':>10} {'Test err':>10} "
      f"{'Gap':>8} {'Rad est':>10}")
print("-" * 85)

for scale, label in [(1.0, "Normal init"), (3.0, "Large init")]:
    for sn_constraint, sn_label in [(None, "unconstrained"), (1.0, "SN<=1.0"), (0.5, "SN<=0.5")]:
        net = make_random_net([d, 32, 16, 1], scale=scale, seed=42)
        if sn_constraint is not None:
            net = project_spectral_norm(net, max_norm=sn_constraint)

        # Predictions (sign of output)
        pred_train = (forward_net(X_train, net) > 0).astype(float)
        pred_test = (forward_net(X_test, net) > 0).astype(float)
        train_err = np.mean(pred_train != y_train)
        test_err = np.mean(pred_test != y_test)
        gap = abs(test_err - train_err)

        sn_prod = spectral_norm_product(net)
        rad_est = mc_rademacher_network(X_train, net)

        config = f"{label}, {sn_label}"
        print(f"{config:<25} {sn_prod:>15.4f} {train_err:>10.4f} {test_err:>10.4f} "
              f"{gap:>8.4f} {rad_est:>10.4f}")

print()
print("Key observations:")
print("1. Constraining spectral norms reduces the spectral-norm product (theory predicts this)")
print("2. Smaller spectral-norm product → smaller Rademacher estimate (as the bound promises)")
print("3. For large-init networks, spectral normalization visibly reduces the generalization gap")
print("4. This is exactly why SNGAN constrains discriminator spectral norms: Rademacher theory")
print("   provides the principled justification for this practical regularization technique.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation → implementation → application → open-ended challenge. Solutions/outlines are in the collapsed `Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a linear class $\{x \mapsto \langle w, x \rangle : \|w\| \leq 2\}$ with $n = 400$ data points on the unit sphere ($\max_i \|x_i\| = 1$), compute the Rademacher complexity upper bound $\hat{\mathcal{R}}_S(\mathcal{F}) \leq B \cdot \max_i \|x_i\| / \sqrt{n}$ by hand, and then the full generalization bound at $\delta = 0.05$.

<details>
<summary>💡 Solution</summary>

The Rademacher upper bound is $\frac{B \cdot \max\|x_i\|}{\sqrt{n}} = \frac{2 \cdot 1}{\sqrt{400}} = \frac{2}{20} = 0.1$.

The generalization bound is $2 \cdot 0.1 + \sqrt{\frac{\ln(2/0.05)}{2 \cdot 400}} = 0.2 + \sqrt{\frac{\ln(40)}{800}} = 0.2 + \sqrt{\frac{3.689}{800}} = 0.2 + \sqrt{0.004611} \approx 0.2 + 0.0679 = 0.268$.

So with 95% confidence, the uniform generalization gap $\sup_{f \in \mathcal{F}} |R(f) - \hat{R}(f)|$ is at most $\approx 0.268$.

</details>

### 💭 UNDERSTAND
2. Explain why $\hat{\mathcal{R}}_S(\mathcal{F}_1) \leq \hat{\mathcal{R}}_S(\mathcal{F}_2)$ whenever $\mathcal{F}_1 \subseteq \mathcal{F}_2$, and connect this to the visualization showing increasing Rademacher complexity with norm bound $B$. Why does this monotonicity make Rademacher complexity a sensible "capacity measure"?

<details>
<summary>💡 Solution</summary>

By definition, $\hat{\mathcal{R}}_S(\mathcal{F}) = \mathbb{E}_\sigma[\sup_{f \in \mathcal{F}} \frac{1}{n}\sum_i \sigma_i f(z_i)]$. The supremum over a subset $\mathcal{F}_1 \subseteq \mathcal{F}_2$ is always at most the supremum over the superset, since maximizing over more candidates can only increase (or maintain) the maximum. This is a pointwise inequality for each $\boldsymbol{\sigma}$, so it passes through the expectation.

In the visualization, increasing $B$ in the linear class $\{\langle w, x \rangle : \|w\| \leq B\}$ strictly enlarges the class (any $w$ with $\|w\| \leq B_1$ also satisfies $\|w\| \leq B_2$ for $B_2 > B_1$), so Rademacher complexity must be monotonically non-decreasing — exactly what the plot shows.

This monotonicity is a minimal desirable property for any capacity measure: a richer class (more functions to choose from) should never appear *less* complex than a restricted subclass. VC dimension also has this property (VC dim of $\mathcal{F}_1 \leq$ VC dim of $\mathcal{F}_2$ when $\mathcal{F}_1 \subseteq \mathcal{F}_2$), so Rademacher complexity is at least as sensible as VC in this regard — and strictly more informative because it also depends on the data.

</details>

### ✏️ DERIVE
3. Derive the Rademacher complexity bound $\hat{\mathcal{R}}_S(\mathcal{F}) \leq B \cdot \max_i \|x_i\| / \sqrt{n}$ for the linear class $\{x \mapsto \langle w, x \rangle : \|w\| \leq B\}$ using only Cauchy–Schwarz and Jensen's inequality (no other tools). Identify exactly where each inequality is used and whether it is tight or loose.

<details>
<summary>💡 Solution</summary>

Starting from the definition:

$$\hat{\mathcal{R}}_S(\mathcal{F}) = \mathbb{E}_\sigma\!\left[\sup_{\|w\| \leq B} \frac{1}{n} \left\langle w, \sum_i \sigma_i x_i \right\rangle\right]$$

**Step 1 (Cauchy–Schwarz):** For fixed $\boldsymbol{\sigma}$, $\sup_{\|w\| \leq B} \langle w, v \rangle = B\|v\|$ (achieved by $w = B \cdot v/\|v\|$). This step is **tight** — Cauchy–Schwarz is an equality when vectors are aligned.

$$= \frac{B}{n} \mathbb{E}_\sigma\!\left[\left\|\sum_i \sigma_i x_i\right\|\right]$$

**Step 2 (Jensen's inequality):** Since $\|\cdot\|$ is concave when composed with $\sqrt{\cdot}$, we apply Jensen to move the expectation inside the square root: $\mathbb{E}[\|v\|] = \mathbb{E}[\sqrt{\|v\|^2}] \leq \sqrt{\mathbb{E}[\|v\|^2]}$. This step is **loose** in general (equality only when $\|v\|$ is deterministic).

$$\leq \frac{B}{n}\sqrt{\mathbb{E}_\sigma\!\left[\left\|\sum_i \sigma_i x_i\right\|^2\right]} = \frac{B}{n}\sqrt{\sum_i \|x_i\|^2} \leq \frac{B}{n}\sqrt{n \max_i \|x_i\|^2} = \frac{B \max_i \|x_i\|}{\sqrt{n}}$$

The cross terms vanish because $\mathbb{E}[\sigma_i \sigma_j] = 0$ for $i \neq j$. The last step replaces average $\|x_i\|^2$ with the maximum, which is **loose** unless all points have equal norm (as on the unit sphere, where it becomes tight).

</details>

### 🐍 IMPLEMENT
4. Extend the Monte Carlo Rademacher estimator to work with an sklearn classifier: given a fitted `sklearn.linear_model.LogisticRegression` model and a dataset $X$, estimate the empirical Rademacher complexity by using the model's `decision_function` as $f$, sampling $\boldsymbol{\sigma}$, and computing $\frac{1}{n}|\sum_i \sigma_i f(x_i)|$ for each draw.

<details>
<summary>💡 Solution</summary>

```python
from sklearn.linear_model import LogisticRegression

def mc_rademacher_sklearn(model, X, n_draws=5000, seed=42):
    rng = np.random.default_rng(seed)
    f_vals = model.decision_function(X)  # shape (n,)
    n = len(f_vals)
    sups = np.zeros(n_draws)
    for t in range(n_draws):
        sigma = rng.choice([-1, 1], size=n)
        sups[t] = abs(np.dot(sigma, f_vals)) / n
    return sups.mean()

# Example:
X_demo = np.random.randn(200, 5)
y_demo = (X_demo[:, 0] > 0).astype(int)
clf = LogisticRegression().fit(X_demo, y_demo)
print(f"Rademacher estimate: {mc_rademacher_sklearn(clf, X_demo):.4f}")
```

Note: this estimates the Rademacher complexity of a *single* function (the fitted model), which lower-bounds the complexity of the entire class. For the full class, you'd need the inner supremum over *all* classifiers in the class, which is what the `empirical_rademacher_linear` function computes analytically for the linear case.

</details>

### 🤖 APPLY
5. Using the implementation code, investigate how **network width** affects Rademacher complexity: create networks with architectures `[d, w, 1]` for widths $w \in \{4, 16, 64, 256\}$ (single hidden layer), with spectral norms projected to $\leq 1$. Measure the Monte Carlo Rademacher estimate and the spectral-norm product for each width. Does wider always mean higher complexity when spectral norms are constrained?

<details>
<summary>💡 Solution</summary>

```python
for width in [4, 16, 64, 256]:
    net = make_random_net([d, width, 1], scale=1.0, seed=42)
    net = project_spectral_norm(net, max_norm=1.0)
    sn_prod = spectral_norm_product(net)
    rad_est = mc_rademacher_network(X_train, net)
    print(f"Width={width:4d}  SN product={sn_prod:.4f}  Rad={rad_est:.4f}")
```

With spectral norms all constrained to $\leq 1$, the spectral-norm product stays $\leq 1$ regardless of width, and the Rademacher estimate remains roughly constant. This is the key insight: **spectral-norm control decouples complexity from width.** A 256-wide network with $\|W\|_{\text{op}} \leq 1$ per layer has essentially the same Rademacher complexity as a 4-wide one. Width only increases complexity when the norms are allowed to grow — which is exactly what happens without explicit regularization.

</details>

### 🚀 CHALLENGE
6. **Two-layer ReLU bound via contraction.** For a network $f(x) = v^\top \text{ReLU}(Wx)$ with $\|v\| \leq B_2$ and $\|W\|_{\text{op}} \leq B_1$, derive a Rademacher complexity bound by:
   (a) Starting from the linear class $\{x \mapsto \langle w, x \rangle : \|w\|\leq B_1\}$ with known complexity $\leq B_1 \max\|x_i\|/\sqrt{n}$,
   (b) Applying the contraction lemma with ReLU ($1$-Lipschitz, ReLU(0)=0),
   (c) Handling the outer linear layer.
   State the final bound and verify it numerically against Monte Carlo estimation.

<details>
<summary>💡 Solution</summary>

**Step (a):** The class $\mathcal{G} = \{x \mapsto Wx : \|W\|_{\text{op}} \leq B_1\}$ produces vector outputs. For each output coordinate $j$, the class of $j$-th coordinates is $\{x \mapsto w_j^\top x : \|w_j\| \leq B_1\}$ (since the $j$-th row of $W$ has norm $\leq \|W\|_{\text{op}} \leq B_1$). Each coordinate's Rademacher complexity is $\leq B_1 \max\|x_i\| / \sqrt{n}$.

**Step (b):** ReLU is $1$-Lipschitz with ReLU(0) = 0, so by the contraction lemma, $\hat{\mathcal{R}}_S(\text{ReLU} \circ \mathcal{G}_j) \leq 1 \cdot \hat{\mathcal{R}}_S(\mathcal{G}_j) \leq B_1 \max\|x_i\| / \sqrt{n}$ for each coordinate.

**Step (c):** The outer layer applies $v^\top$ with $\|v\| \leq B_2$. By Cauchy–Schwarz over coordinates and summing Rademacher complexities:

$$\hat{\mathcal{R}}_S(\mathcal{F}_{\text{net}}) \leq \frac{B_1 \cdot B_2 \cdot \max_i \|x_i\|}{\sqrt{n}}$$

This is exactly the spectral-norm product bound specialized to two layers. For $B_1 = B_2 = 1$, $\max\|x_i\| = 1$, $n = 200$: bound $= 1/\sqrt{200} \approx 0.0707$. Verify with `mc_rademacher_network` on a spectrally-projected two-layer net.

</details>

---

## 📚 Further Reading
- Bartlett & Mendelson, ["Rademacher and Gaussian Complexities: Risk Bounds and Structural Results"](https://www.jmlr.org/papers/volume3/bartlett02a/bartlett02a.pdf), *JMLR*, 2002
- Mohri, Rostamizadeh & Talwalkar, *Foundations of Machine Learning*, Ch. 3 (Rademacher Complexity)
- Bartlett, Foster & Telgarsky, ["Spectrally-normalized margin bounds for neural networks"](https://arxiv.org/abs/1706.08498), NeurIPS 2017 — *theorem source for the spectral-norm-product bound*
- Neyshabur, Bhojanapalli, McAllester & Srebro, ["A PAC-Bayesian Approach to Spectrally-Normalized Margin Bounds for Neural Networks"](https://arxiv.org/abs/1707.09564), ICLR 2018
- Dziugaite & Roy, ["Computing Nonvacuous Generalization Bounds for Deep (Stochastic) Neural Networks"](https://arxiv.org/abs/1703.11008), UAI 2017 — *first non-vacuous bounds for such networks*
- Neyshabur, Bhojanapalli, McAllester & Srebro, ["Exploring Generalization in Deep Nets"](https://arxiv.org/abs/1706.08947), NeurIPS 2017
- Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, Ch. 26 (Rademacher Complexity)
- Miyato, Kataoka, Koyama & Yoshida, ["Spectral Normalization for GANs"](https://arxiv.org/abs/1802.05957), ICLR 2018

---

*Next notebook: `13.07` Algorithmic Stability and Generalization*